# Qwen3-1.7B — SCS-LoRA Prompt — Cascade Scale 2.0 — BF16

Notebook **train adapter only**.

Giữ nguyên type-aware prompt của notebook gốc; chỉ thay `CASCADED_INPUT_SCALE = 2.0` để làm ablation.

Các cấu hình còn lại giữ nguyên: BF16, backbone frozen, batch=4, eval batch=4,
gradient accumulation=4, 3 epochs, LR=2e-5, warmup=5%, linear scheduler,
eval mỗi 400 steps, early stopping patience=2, r=8, alpha=16, dropout=0.05,
target q/k/v/o, gold one-hot routing, complete adapter save check.

In [ ]:
# Optional: chỉ chạy nếu môi trường chưa có đủ package.
# %pip install -q transformers accelerate datasets pandas numpy torch tqdm sentencepiece protobuf scikit-learn

In [ ]:
import datetime as dt
import inspect
import json
import math
import os
import platform
import random
import sys
from pathlib import Path
from typing import Optional

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import Dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    Trainer,
    TrainerCallback,
    TrainingArguments,
    set_seed,
)

os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

In [ ]:
# ============================================================
# 1. MODEL-SPECIFIC SETTINGS
# ============================================================
BACKBONE_MODEL = 'HTThuanHcmus/qwen3-1.7b-merge'
TOKENIZER_MODEL = 'HTThuanHcmus/qwen3-1.7b-merge'
TOKENIZER_USE_FAST = True
MODEL_FAMILY = 'qwen'

VARIANT_NAME = 'qwen3_1_7b_scs_lora_prompt_scale_2_0'
ADAPTER_FILENAME = 'qwen3_1_7b_scs_lora_prompt_scale_2_0_best_adapter.pt'
EXPECTED_INJECTED_MODULES = 112

In [ ]:
# ============================================================
# 2. COMMON TRAINING CONFIGURATION
# ============================================================
SEED = 42
BF16_DTYPE = torch.bfloat16

# Gold question type is read directly from train.csv / val.csv.
TYPE_ORDER = ["COMPARISON", "FACTOID", "SUMMARY", "VERIFICATION"]
TYPE2ID = {name: idx for idx, name in enumerate(TYPE_ORDER)}
NUM_EXPERTS = len(TYPE_ORDER)

LORA_RANK = 8
LORA_ALPHA = 16
LORA_DROPOUT = 0.05
TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj"]
CASCADED_INPUT_SCALE = 2.0

MAX_INPUT_LEN = 1500
MAX_TARGET_LEN = 256
TRAIN_BATCH_SIZE = 4
EVAL_BATCH_SIZE = 4
GRAD_ACCUM = 4
EFFECTIVE_BATCH_SIZE_PER_PROCESS = TRAIN_BATCH_SIZE * GRAD_ACCUM

NUM_EPOCHS = 3
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.0
WARMUP_RATIO = 0.05
LOGGING_STEPS = 10
EVAL_STEPS = 400
EARLY_STOPPING_PATIENCE = 2
EARLY_STOPPING_THRESHOLD = 0.0

TYPE_INSTRUCTIONS = {
    "COMPARISON": "Trả lời bằng cách so sánh rõ ràng. Giữ đúng chiều tăng/giảm, cao hơn/thấp hơn và các số liệu liên quan.",
    "FACTOID": "Trả lời ngắn gọn, trực tiếp. Giữ nguyên số liệu, đơn vị, tên công ty và thực thể tài chính trong ngữ cảnh.",
    "SUMMARY": "Tóm tắt hoặc diễn giải ngắn gọn dựa trên ngữ cảnh. Không tự tạo thông tin ngoài ngữ cảnh.",
    "VERIFICATION": "Trả lời theo hướng xác minh. Nếu phù hợp, bắt đầu bằng Đúng hoặc Không đúng, sau đó nêu bằng chứng ngắn gọn.",
}
PROMPT_SUFFIX = "\ntrả lời: "

In [ ]:
# ============================================================
# 3. PATHS + PERSISTENT LOGGING
# ============================================================
# Folder layout expected:
# .
# ├── this_notebook.ipynb
# └── data/
#     ├── train.csv
#     └── val.csv
TRAIN_PATH = Path("data/train.csv")
VAL_PATH = Path("data/val.csv")

OUTPUT_DIR = Path(f"outputs-{VARIANT_NAME}")
TOKENIZER_OUTPUT_DIR = OUTPUT_DIR / "tokenizer"
ADAPTER_PATH = OUTPUT_DIR / ADAPTER_FILENAME
EVAL_METRICS_PATH = OUTPUT_DIR / f"{VARIANT_NAME}_eval_metrics.json"
RUN_CONFIG_PATH = OUTPUT_DIR / f"{VARIANT_NAME}_run_config.json"
TRAINER_LOG_HISTORY_PATH = OUTPUT_DIR / f"{VARIANT_NAME}_trainer_log_history.json"
ENVIRONMENT_INFO_PATH = OUTPUT_DIR / f"{VARIANT_NAME}_environment.json"
TRAINER_OUTPUT_DIR = OUTPUT_DIR / "trainer"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if not TRAIN_PATH.is_file():
    raise FileNotFoundError(f"Missing training file: {TRAIN_PATH.resolve()}")
if not VAL_PATH.is_file():
    raise FileNotFoundError(f"Missing validation file: {VAL_PATH.resolve()}")

LOG_DIR = OUTPUT_DIR / "logs"
LOG_DIR.mkdir(parents=True, exist_ok=True)
LOG_FILE = LOG_DIR / f'run_{dt.datetime.now().strftime("%Y%m%d_%H%M%S")}.log'
RUN_LOG_PATH = LOG_FILE

class Tee:
    def __init__(self, *files):
        self.files = files

    def write(self, text):
        for f in self.files:
            f.write(text)
            f.flush()

    def flush(self):
        for f in self.files:
            f.flush()

    def isatty(self):
        return False

log_fp = open(LOG_FILE, "w", encoding="utf-8")
sys.stdout = Tee(sys.__stdout__, log_fp)
sys.stderr = Tee(sys.__stderr__, log_fp)

print(f"Logging to: {LOG_FILE}")
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# ============================================================
# 4. STRICT CUDA / BF16 PREFLIGHT
# ============================================================
def require_cuda_bf16() -> torch.device:
    if not torch.cuda.is_available():
        raise RuntimeError(
            "CUDA is unavailable. CPU/FP32 fallback is disabled for this experiment."
        )
    if not torch.cuda.is_bf16_supported():
        raise RuntimeError(
            "Visible GPU does not support BF16. FP16/FP32 fallback is disabled."
        )
    device = torch.device("cuda:0")
    torch.cuda.set_device(device)
    return device

DEVICE = require_cuda_bf16()

def seed_everything(seed: int) -> None:
    set_seed(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(SEED)

def trainable_dtype_set(model: nn.Module):
    return sorted({str(p.dtype) for p in model.parameters() if p.requires_grad})

def assert_all_trainable_bf16(model: nn.Module) -> None:
    dtypes = trainable_dtype_set(model)
    print("Trainable parameter dtypes:", dtypes)
    if dtypes != ["torch.bfloat16"]:
        raise RuntimeError(
            f"All trainable adapter parameters must be BF16; got {dtypes}."
        )

In [ ]:
# ============================================================
# 5. DATA + TRAINING PROMPT + AUTOTOKENIZER
# ============================================================
def canonical_question_type(value) -> str:
    text = str(value).strip().upper().replace("-", "_").replace(" ", "_")
    aliases = {
        "FACTOID_EXTRACTION": "FACTOID",
        "SUMMARY_INTERPRETATION": "SUMMARY",
        "SUMMARY_AND_INTERPRETATION": "SUMMARY",
    }
    text = aliases.get(text, text)
    if text not in TYPE2ID:
        raise ValueError(f"Unknown question_type={value!r}; expected one of {TYPE_ORDER}")
    return text

def load_training_csv(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path)
    df.columns = [str(c).strip().lower() for c in df.columns]
    required = {"question", "context", "answer", "question_type"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(
            f"{path} missing required columns {sorted(missing)}; "
            f"found {df.columns.tolist()}"
        )

    for column in ["question", "context", "answer"]:
        df[column] = df[column].astype(str).str.strip()

    df["question_type"] = df["question_type"].map(canonical_question_type)
    df = df.replace({"nan": "", "None": ""})
    df = df[
        df["question"].str.len().gt(0)
        & df["context"].str.len().gt(0)
        & df["answer"].str.len().gt(0)
    ]
    return df.reset_index(drop=True)

def build_training_input(question: str, context: str, question_type: str) -> str:
    qt = canonical_question_type(question_type)
    instruction = TYPE_INSTRUCTIONS[qt]
    return (
        f"loại câu hỏi: {qt}\n"
        f"hướng dẫn: {instruction}\n"
        f"câu hỏi: {str(question).strip()}\n"
        f"ngữ cảnh: {str(context).strip()}"
    )

def load_training_tokenizer():
    if MODEL_FAMILY == "llama":
        # AutoTokenizer only; Llama explicitly uses the slow tokenizer.
        tokenizer = AutoTokenizer.from_pretrained(
            TOKENIZER_MODEL,
            use_fast=False,
            trust_remote_code=True,
        )
    else:
        # AutoTokenizer only. Fallbacks handle tokenizer metadata differences
        # without switching tokenizer classes manually.
        attempts = [
            dict(use_fast=TOKENIZER_USE_FAST, trust_remote_code=True, extra_special_tokens={}),
            dict(use_fast=TOKENIZER_USE_FAST, trust_remote_code=True),
            dict(use_fast=False, trust_remote_code=True, extra_special_tokens={}),
            dict(use_fast=False, trust_remote_code=True),
        ]
        tokenizer = None
        last_error = None
        for kwargs in attempts:
            try:
                tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_MODEL, **kwargs)
                print("Loaded tokenizer with AutoTokenizer:", kwargs)
                break
            except Exception as exc:
                last_error = exc
                print("Tokenizer load attempt failed:", kwargs, repr(exc))
        if tokenizer is None:
            raise RuntimeError(
                f"Cannot load tokenizer from {TOKENIZER_MODEL}: {last_error}"
            )

    if tokenizer.eos_token_id is None:
        raise RuntimeError(f"Tokenizer {TOKENIZER_MODEL!r} has no eos_token_id")
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token

    tokenizer.padding_side = "right"
    print("Tokenizer class:", tokenizer.__class__.__name__)
    print("Tokenizer source:", TOKENIZER_MODEL)
    print("Tokenizer is_fast:", bool(getattr(tokenizer, "is_fast", False)))

    if MODEL_FAMILY == "llama" and bool(getattr(tokenizer, "is_fast", False)):
        raise RuntimeError(
            "Llama tokenizer must be loaded with AutoTokenizer(use_fast=False)."
        )
    return tokenizer

In [ ]:
# ============================================================
# 6. SCS-LORA MODULES — TRAINING ONLY, GOLD ONE-HOT ROUTING
# ============================================================
class LoRAExpert(nn.Module):
    def __init__(self, in_features: int, out_features: int):
        super().__init__()
        self.scaling = LORA_ALPHA / LORA_RANK
        self.dropout = nn.Dropout(LORA_DROPOUT)
        self.lora_A = nn.Linear(
            in_features, LORA_RANK, bias=False, dtype=BF16_DTYPE
        )
        self.lora_B = nn.Linear(
            LORA_RANK, out_features, bias=False, dtype=BF16_DTYPE
        )
        nn.init.kaiming_uniform_(self.lora_A.weight, a=math.sqrt(5))
        nn.init.zeros_(self.lora_B.weight)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if x.dtype != BF16_DTYPE:
            x = x.to(dtype=BF16_DTYPE)
        return self.lora_B(self.lora_A(self.dropout(x))) * self.scaling

class SCSLoRALinear(nn.Module):
    def __init__(self, base_layer: nn.Linear):
        super().__init__()
        self.base_layer = base_layer
        self.in_features = base_layer.in_features
        self.out_features = base_layer.out_features
        self.training_route_weights: Optional[torch.Tensor] = None

        # Base projection remains frozen.
        for parameter in self.base_layer.parameters():
            parameter.requires_grad = False

        self.experts = nn.ModuleList([
            LoRAExpert(self.in_features, self.out_features)
            for _ in range(NUM_EXPERTS)
        ])
        self.shared_out_expert = LoRAExpert(
            self.in_features, self.out_features
        )
        self.shared_input_expert = LoRAExpert(
            self.in_features, self.in_features
        )

    def set_training_route_weights(
        self, weights: Optional[torch.Tensor]
    ) -> None:
        self.training_route_weights = weights

    def _route_gold_experts(
        self, x: torch.Tensor, alpha: torch.Tensor
    ) -> torch.Tensor:
        if alpha.dim() == 1:
            alpha = alpha.unsqueeze(0)

        if alpha.shape[0] != x.shape[0]:
            if x.shape[0] % alpha.shape[0] != 0:
                raise RuntimeError(
                    f"Cannot expand routing batch {alpha.shape[0]} "
                    f"to hidden batch {x.shape[0]}"
                )
            alpha = alpha.repeat_interleave(
                x.shape[0] // alpha.shape[0], dim=0
            )

        alpha = alpha.to(device=x.device, dtype=BF16_DTYPE)
        route_ids = alpha.argmax(dim=-1)

        out = x.new_zeros(
            (*x.shape[:-1], self.out_features), dtype=BF16_DTYPE
        )
        for expert_id, expert in enumerate(self.experts):
            indices = torch.nonzero(
                route_ids == expert_id, as_tuple=False
            ).flatten()
            if indices.numel() == 0:
                continue
            selected_x = torch.index_select(x, 0, indices)
            selected_out = expert(selected_x)
            out.index_copy_(0, indices, selected_out)
        return out

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        base_out = self.base_layer(x)
        shared_out = self.shared_out_expert(x)
        expert_input = (
            x
            + CASCADED_INPUT_SCALE
            * self.shared_input_expert(x)
        )

        if self.training_route_weights is None:
            raise RuntimeError(
                "SCS-LoRA training route is unset. "
                "Gold one-hot question_type routing is required."
            )

        type_out = self._route_gold_experts(
            expert_input, self.training_route_weights
        )
        return base_out + shared_out + type_out

def freeze_backbone(model: nn.Module) -> None:
    for parameter in model.parameters():
        parameter.requires_grad = False

def _get_parent_module(model: nn.Module, module_name: str):
    parts = module_name.split(".")
    parent = model
    for part in parts[:-1]:
        parent = getattr(parent, part)
    return parent, parts[-1]

def inject_scs_lora(model: nn.Module) -> nn.Module:
    replaced = []
    for name, module in list(model.named_modules()):
        if (
            isinstance(module, nn.Linear)
            and name.split(".")[-1] in TARGET_MODULES
        ):
            parent, child_name = _get_parent_module(model, name)
            setattr(parent, child_name, SCSLoRALinear(module))
            replaced.append(name)

    print(f"Injected SCS-LoRA into {len(replaced)} linear modules")
    if len(replaced) != EXPECTED_INJECTED_MODULES:
        raise RuntimeError(
            f"Expected {EXPECTED_INJECTED_MODULES} target modules "
            f"for {BACKBONE_MODEL}, but replaced {len(replaced)}. "
            "Stop: backbone architecture/revision may have changed."
        )
    return model

def set_training_route_weights(
    model: nn.Module, weights: Optional[torch.Tensor]
) -> None:
    for module in model.modules():
        if isinstance(module, SCSLoRALinear):
            module.set_training_route_weights(weights)

def is_scs_adapter_parameter(name: str) -> bool:
    return any(
        token in name
        for token in (
            ".experts.",
            ".shared_out_expert.",
            ".shared_input_expert.",
        )
    )

def assert_backbone_frozen(model: nn.Module) -> None:
    unexpected_trainable = [
        name
        for name, parameter in model.named_parameters()
        if parameter.requires_grad
        and not is_scs_adapter_parameter(name)
    ]
    if unexpected_trainable:
        raise RuntimeError(
            "Backbone is not fully frozen. Unexpected trainable "
            f"parameters: {unexpected_trainable[:20]}"
        )

    adapter_trainable = [
        name
        for name, parameter in model.named_parameters()
        if parameter.requires_grad
        and is_scs_adapter_parameter(name)
    ]
    if not adapter_trainable:
        raise RuntimeError("No trainable SCS-LoRA parameters found.")

    print("Backbone frozen: YES")
    print("Trainable SCS-LoRA parameter tensors:", len(adapter_trainable))

def count_parameters(model: nn.Module):
    trainable = sum(
        p.numel() for p in model.parameters() if p.requires_grad
    )
    total = sum(p.numel() for p in model.parameters())
    print(
        f"Trainable parameters: {trainable:,} / {total:,} "
        f"({100.0 * trainable / total:.4f}%)"
    )
    return trainable, total

In [ ]:
# ============================================================
# 7. DATASET / COLLATOR / TRAINER
# ============================================================
class SCSQADataset(Dataset):
    def __init__(self, dataframe: pd.DataFrame, tokenizer):
        self.data = dataframe.reset_index(drop=True)
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):
        row = self.data.iloc[index]
        qt = canonical_question_type(row["question_type"])

        prompt = (
            build_training_input(
                row["question"], row["context"], qt
            )
            + PROMPT_SUFFIX
        )
        answer = str(row["answer"]).strip()

        if self.tokenizer.eos_token is not None:
            answer = answer + self.tokenizer.eos_token

        # Preserve the established train tokenization semantics.
        prompt_ids = self.tokenizer(
            prompt,
            add_special_tokens=True,
            truncation=True,
            max_length=MAX_INPUT_LEN,
        )["input_ids"]

        answer_ids = self.tokenizer(
            answer,
            add_special_tokens=False,
            truncation=True,
            max_length=MAX_TARGET_LEN,
        )["input_ids"]

        input_ids = prompt_ids + answer_ids
        labels = [-100] * len(prompt_ids) + answer_ids

        route_weights = torch.zeros(
            NUM_EXPERTS, dtype=BF16_DTYPE
        )
        route_weights[TYPE2ID[qt]] = 1

        return {
            "input_ids": torch.tensor(
                input_ids, dtype=torch.long
            ),
            "labels": torch.tensor(
                labels, dtype=torch.long
            ),
            "route_weights": route_weights,
        }

class SCSCollator:
    def __init__(self, pad_token_id: int):
        self.pad_token_id = pad_token_id

    def __call__(self, features):
        input_ids = pad_sequence(
            [f["input_ids"] for f in features],
            batch_first=True,
            padding_value=self.pad_token_id,
        )
        labels = pad_sequence(
            [f["labels"] for f in features],
            batch_first=True,
            padding_value=-100,
        )
        # Build attention mask from original sequence lengths.
        # This remains correct when pad_token_id == eos_token_id.
        attention_mask = pad_sequence(
            [
                torch.ones(
                    f["input_ids"].shape[0],
                    dtype=torch.long,
                )
                for f in features
            ],
            batch_first=True,
            padding_value=0,
        )
        route_weights = torch.stack(
            [f["route_weights"] for f in features], dim=0
        )

        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": labels,
            "route_weights": route_weights,
        }

class SCSTrainer(Trainer):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)

        # compute_loss below returns a mean-reduced micro-batch loss and does not
        # consume num_items_in_batch. Explicitly tell Trainer to apply gradient-
        # accumulation scaling itself. This is safe on older versions too.
        self.model_accepts_loss_kwargs = False

    @staticmethod
    def _validate_gold_one_hot(route_weights: torch.Tensor) -> None:
        if route_weights.dim() != 2 or route_weights.shape[1] != NUM_EXPERTS:
            raise RuntimeError(
                f"route_weights must have shape [batch, {NUM_EXPERTS}], "
                f"got {tuple(route_weights.shape)}"
            )

        row_sums = route_weights.sum(dim=-1)
        max_vals, _ = route_weights.max(dim=-1)
        valid = bool(
            torch.all(row_sums == 1).item()
            and torch.all(max_vals == 1).item()
            and torch.all((route_weights == 0) | (route_weights == 1)).item()
        )
        if not valid:
            raise RuntimeError(
                "Training routing must be exact gold one-hot for every sample."
            )

    def compute_loss(
        self,
        model,
        inputs,
        return_outputs=False,
        **kwargs,
    ):
        route_weights = inputs.pop("route_weights")
        self._validate_gold_one_hot(route_weights)

        route_weights = route_weights.to(
            device=model.device,
            dtype=BF16_DTYPE,
        )

        try:
            set_training_route_weights(model, route_weights)
            outputs = model(**inputs)
            loss = outputs.loss

            if not torch.isfinite(loss):
                raise RuntimeError(f"Non-finite training loss: {loss}")
        finally:
            set_training_route_weights(model, None)

        return (loss, outputs) if return_outputs else loss


In [ ]:
# ============================================================
# 8. COMPLETE ADAPTER SAVE / LOAD + EARLY STOPPING
# ============================================================
def scs_state_dict(model: nn.Module):
    state = {}
    for key, value in model.state_dict().items():
        if any(
            token in key
            for token in (
                "experts.",
                "shared_out_expert",
                "shared_input_expert",
            )
        ):
            tensor = (
                value.detach()
                .to(device="cpu")
                .contiguous()
            )
            if tensor.dtype != BF16_DTYPE:
                raise RuntimeError(
                    f"Non-BF16 adapter tensor during save: "
                    f"{key} -> {tensor.dtype}"
                )
            state[key] = tensor

    if not state:
        raise RuntimeError(
            "No SCS-LoRA tensors were found for saving."
        )
    return state

def save_scs_adapter(
    model: nn.Module,
    path: Path,
    trainable: int,
    total: int,
    best_step=None,
    best_eval_loss=None,
):
    state = scs_state_dict(model)
    saved_numel = sum(t.numel() for t in state.values())

    if saved_numel != trainable:
        raise RuntimeError(
            "Incomplete SCS-LoRA adapter save: "
            f"state_dict contains {saved_numel:,} parameters "
            f"but model reports {trainable:,} trainable parameters."
        )

    payload = {
        "format": "scs_lora_adapter_v1",
        "state_dict": state,
        "meta": {
            "variant_name": VARIANT_NAME,
            "method": "scs_lora",
            "backbone_model": BACKBONE_MODEL,
            "tokenizer_model": TOKENIZER_MODEL,
            "tokenizer_use_fast": TOKENIZER_USE_FAST,
            "type_order": TYPE_ORDER,
            "training_routing": "gold_one_hot",
            "training_prompt": "type_aware",
            "lora_rank": LORA_RANK,
            "lora_alpha": LORA_ALPHA,
            "lora_dropout": LORA_DROPOUT,
            "target_modules": TARGET_MODULES,
            "cascaded_input_scale": CASCADED_INPUT_SCALE,
            "shared_output_adapter": True,
            "shared_input_adapter": True,
            "precision": "bfloat16",
            "best_step": best_step,
            "best_eval_loss": best_eval_loss,
            "adapter_parameter_count": saved_numel,
            "trainable_parameters": trainable,
            "total_parameters": total,
        },
    }

    path.parent.mkdir(
        parents=True, exist_ok=True
    )
    torch.save(payload, path)
    print(
        "Saved complete best SCS-LoRA adapter:",
        path,
    )
    print(
        "Saved adapter parameters:",
        f"{saved_numel:,}",
    )

def load_scs_adapter(
    model: nn.Module, path: Path
):
    try:
        payload = torch.load(
            path,
            map_location="cpu",
            weights_only=False,
        )
    except TypeError:
        payload = torch.load(
            path, map_location="cpu"
        )

    missing, unexpected = model.load_state_dict(
        payload["state_dict"],
        strict=False,
    )
    if unexpected:
        raise RuntimeError(
            f"Unexpected SCS adapter keys: "
            f"{unexpected[:20]}"
        )

    print(
        "Reloaded best SCS-LoRA adapter:",
        path,
    )
    print(
        "Missing base-model keys "
        "(expected because only adapter is loaded):",
        len(missing),
    )
    return payload

class BestAdapterEarlyStopCallback(
    TrainerCallback
):
    def __init__(
        self,
        adapter_path: Path,
        trainable: int,
        total: int,
    ):
        self.adapter_path = adapter_path
        self.trainable = trainable
        self.total = total
        self.best_loss = None
        self.best_step = None
        self.bad_evals = 0

    def on_evaluate(
        self,
        args,
        state,
        control,
        metrics=None,
        model=None,
        **kwargs,
    ):
        metrics = metrics or {}
        if (
            model is None
            or "eval_loss" not in metrics
        ):
            return control

        loss = float(metrics["eval_loss"])
        improved = (
            self.best_loss is None
            or loss
            < self.best_loss
            - EARLY_STOPPING_THRESHOLD
        )

        if improved:
            self.best_loss = loss
            self.best_step = int(
                state.global_step
            )
            self.bad_evals = 0

            save_scs_adapter(
                model,
                self.adapter_path,
                self.trainable,
                self.total,
                best_step=self.best_step,
                best_eval_loss=self.best_loss,
            )
            print(
                f"New best eval_loss="
                f"{loss:.6f} at step="
                f"{state.global_step}"
            )
        else:
            self.bad_evals += 1
            print(
                "No eval_loss improvement: "
                f"{self.bad_evals}/"
                f"{EARLY_STOPPING_PATIENCE}"
            )

            if (
                self.bad_evals
                >= EARLY_STOPPING_PATIENCE
            ):
                control.should_training_stop = True
                print(
                    "Early stopping triggered."
                )

        return control

In [ ]:
# ============================================================
# 9. MODEL + TRAINER BUILD
# ============================================================
def load_backbone_bf16():
    common = dict(
        low_cpu_mem_usage=True,
        trust_remote_code=True,
    )

    try:
        model = AutoModelForCausalLM.from_pretrained(
            BACKBONE_MODEL,
            dtype=BF16_DTYPE,
            **common,
        )
    except TypeError:
        model = AutoModelForCausalLM.from_pretrained(
            BACKBONE_MODEL,
            torch_dtype=BF16_DTYPE,
            **common,
        )

    return model

def build_model():
    model = load_backbone_bf16()

    # Explicitly freeze the complete pretrained backbone first.
    freeze_backbone(model)

    # Inject fresh trainable SCS-LoRA branches only after freezing.
    model = inject_scs_lora(model)
    model.to(
        device=DEVICE,
        dtype=BF16_DTYPE,
    )
    model.config.use_cache = False

    assert_backbone_frozen(model)
    assert_all_trainable_bf16(model)
    trainable, total = count_parameters(model)
    return model, trainable, total

def build_training_arguments():
    kwargs = dict(
        output_dir=str(TRAINER_OUTPUT_DIR),
        per_device_train_batch_size=TRAIN_BATCH_SIZE,
        per_device_eval_batch_size=EVAL_BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        num_train_epochs=NUM_EPOCHS,
        learning_rate=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
        lr_scheduler_type="linear",
        logging_steps=LOGGING_STEPS,
        eval_steps=EVAL_STEPS,
        save_strategy="no",
        load_best_model_at_end=False,
        report_to="none",
        remove_unused_columns=False,
        bf16=True,
        fp16=False,
        dataloader_pin_memory=False,
        seed=SEED,
        data_seed=SEED,
    )

    signature = inspect.signature(
        TrainingArguments.__init__
    )

    # Transformers compatibility while preserving the same 5% warmup.
    if "warmup_ratio" in signature.parameters:
        # Transformers v4-style API.
        kwargs["warmup_ratio"] = WARMUP_RATIO
        warmup_api = "warmup_ratio"
    elif "warmup_steps" in signature.parameters:
        # Transformers v5: float warmup_steps in [0, 1) is interpreted as a ratio.
        kwargs["warmup_steps"] = WARMUP_RATIO
        warmup_api = "warmup_steps(float ratio)"
    else:
        raise RuntimeError(
            "TrainingArguments exposes neither warmup_ratio nor warmup_steps; "
            "cannot preserve warmup=0.05."
        )

    print(
        f"Warmup configuration: {warmup_api}={WARMUP_RATIO} "
        "(5% of total training steps)"
    )
    if (
        "eval_strategy"
        in signature.parameters
    ):
        kwargs["eval_strategy"] = "steps"
    else:
        kwargs["evaluation_strategy"] = "steps"

    return TrainingArguments(**kwargs)

def build_trainer(
    model,
    args,
    train_ds,
    val_ds,
    collator,
    tokenizer,
    callback,
):
    kwargs = dict(
        model=model,
        args=args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        data_collator=collator,
        callbacks=[callback],
    )

    signature = inspect.signature(
        Trainer.__init__
    )
    if (
        "processing_class"
        in signature.parameters
    ):
        kwargs["processing_class"] = tokenizer
    else:
        kwargs["tokenizer"] = tokenizer

    return SCSTrainer(**kwargs)

In [ ]:
# ============================================================
# 10. TRAIN
# ============================================================
def main():
    print("=" * 88)
    print("SCS-LoRA TRAIN ONLY — BF16")
    print("=" * 88)
    print("Timestamp:", dt.datetime.now().isoformat())
    print("Python:", platform.python_version())
    print("PyTorch:", torch.__version__)
    print("CUDA runtime:", torch.version.cuda)
    print("GPU:", torch.cuda.get_device_name(DEVICE))
    print("BF16 supported:", torch.cuda.is_bf16_supported())
    print("Backbone:", BACKBONE_MODEL)
    print(
        "Tokenizer:",
        TOKENIZER_MODEL,
        "| use_fast=",
        TOKENIZER_USE_FAST,
    )
    print("Train path:", TRAIN_PATH)
    print("Val path:", VAL_PATH)
    print("Output:", OUTPUT_DIR)
    print("Persistent log:", RUN_LOG_PATH)
    print(
        "Train batch:",
        TRAIN_BATCH_SIZE,
        "| Eval batch:",
        EVAL_BATCH_SIZE,
        "| Grad accumulation:",
        GRAD_ACCUM,
        "| Effective batch/process:",
        EFFECTIVE_BATCH_SIZE_PER_PROCESS,
    )
    print("Training prompt:", "type_aware")
    print("Cascade input scale:", CASCADED_INPUT_SCALE)
    print(
        "Training routing: gold one-hot question_type"
    )
    print("Backbone trainable: NO")
    print(
        "IMPORTANT: run only ONE heavy training "
        "process per physical GPU."
    )

    train_df = load_training_csv(TRAIN_PATH)
    val_df = load_training_csv(VAL_PATH)

    print(
        "Train rows:",
        len(train_df),
        "| Val rows:",
        len(val_df),
    )
    print(
        "Train question-type distribution:\n",
        train_df["question_type"].value_counts(),
    )

    tokenizer = load_training_tokenizer()
    tokenizer.save_pretrained(
        TOKENIZER_OUTPUT_DIR
    )

    model, trainable, total = build_model()

    train_ds = SCSQADataset(
        train_df, tokenizer
    )
    val_ds = SCSQADataset(
        val_df, tokenizer
    )
    collator = SCSCollator(
        tokenizer.pad_token_id
    )

    callback = BestAdapterEarlyStopCallback(
        ADAPTER_PATH,
        trainable,
        total,
    )

    trainer = build_trainer(
        model,
        build_training_arguments(),
        train_ds,
        val_ds,
        collator,
        tokenizer,
        callback,
    )

    trainer.train()

    # Ensure at least one evaluated adapter is saved.
    if callback.best_loss is None:
        metrics = trainer.evaluate()
        if not ADAPTER_PATH.is_file():
            save_scs_adapter(
                trainer.model,
                ADAPTER_PATH,
                trainable,
                total,
                best_step=int(
                    trainer.state.global_step
                ),
                best_eval_loss=float(
                    metrics["eval_loss"]
                ),
            )

    if not ADAPTER_PATH.is_file():
        raise RuntimeError(
            "Training ended without producing "
            "the required SCS-LoRA adapter."
        )

    # Reload the best adapter only to validate/save training results.
    best_payload = load_scs_adapter(
        trainer.model,
        ADAPTER_PATH,
    )
    assert_backbone_frozen(trainer.model)
    assert_all_trainable_bf16(trainer.model)

    final_metrics = trainer.evaluate()
    print(
        "Best-adapter validation metrics:",
        final_metrics,
    )

    with EVAL_METRICS_PATH.open(
        "w", encoding="utf-8"
    ) as f:
        json.dump(
            final_metrics,
            f,
            indent=2,
            ensure_ascii=False,
        )

    with TRAINER_LOG_HISTORY_PATH.open(
        "w", encoding="utf-8"
    ) as f:
        json.dump(
            trainer.state.log_history,
            f,
            indent=2,
            ensure_ascii=False,
        )

    run_config = {
        "variant_name": VARIANT_NAME,
        "method": "scs_lora",
        "backbone_model": BACKBONE_MODEL,
        "tokenizer_model": TOKENIZER_MODEL,
        "tokenizer_use_fast": TOKENIZER_USE_FAST,
        "train_path": str(TRAIN_PATH),
        "val_path": str(VAL_PATH),
        "adapter_path": str(ADAPTER_PATH),
        "precision": "bfloat16_only",
        "backbone_frozen": True,
        "type_order": TYPE_ORDER,
        "training_routing": "gold_one_hot",
        "training_prompt": "type_aware",
        "cascaded_input_scale": CASCADED_INPUT_SCALE,
        "shared_output_adapter": True,
        "shared_input_adapter": True,
        "lora_rank": LORA_RANK,
        "lora_alpha": LORA_ALPHA,
        "lora_dropout": LORA_DROPOUT,
        "target_modules": TARGET_MODULES,
        "max_input_len": MAX_INPUT_LEN,
        "max_target_len": MAX_TARGET_LEN,
        "train_batch_size": TRAIN_BATCH_SIZE,
        "eval_batch_size": EVAL_BATCH_SIZE,
        "gradient_accumulation_steps": GRAD_ACCUM,
        "effective_batch_size_per_process":
            EFFECTIVE_BATCH_SIZE_PER_PROCESS,
        "num_epochs": NUM_EPOCHS,
        "learning_rate": LEARNING_RATE,
        "weight_decay": WEIGHT_DECAY,
        "warmup_ratio": WARMUP_RATIO,
        "eval_steps": EVAL_STEPS,
        "early_stopping_patience":
            EARLY_STOPPING_PATIENCE,
        "early_stopping_threshold":
            EARLY_STOPPING_THRESHOLD,
        "seed": SEED,
        "best_step":
            best_payload["meta"].get("best_step"),
        "best_eval_loss":
            best_payload["meta"].get("best_eval_loss"),
        "adapter_parameter_count":
            best_payload["meta"].get(
                "adapter_parameter_count"
            ),
        "trainable_parameters": trainable,
        "total_parameters": total,
    }

    with RUN_CONFIG_PATH.open(
        "w", encoding="utf-8"
    ) as f:
        json.dump(
            run_config,
            f,
            indent=2,
            ensure_ascii=False,
        )

    environment = {
        "timestamp":
            dt.datetime.now().isoformat(),
        "python":
            platform.python_version(),
        "pytorch":
            torch.__version__,
        "cuda_runtime":
            torch.version.cuda,
        "gpu":
            torch.cuda.get_device_name(DEVICE),
        "bf16_supported":
            torch.cuda.is_bf16_supported(),
        "trainable_dtypes":
            trainable_dtype_set(trainer.model),
    }

    with ENVIRONMENT_INFO_PATH.open(
        "w", encoding="utf-8"
    ) as f:
        json.dump(
            environment,
            f,
            indent=2,
            ensure_ascii=False,
        )

    print("Saved adapter:", ADAPTER_PATH)
    print("Saved tokenizer:", TOKENIZER_OUTPUT_DIR)
    print("Saved metrics:", EVAL_METRICS_PATH)
    print("Saved run config:", RUN_CONFIG_PATH)
    print(
        "Saved trainer history:",
        TRAINER_LOG_HISTORY_PATH,
    )
    print("Training complete.")

if __name__ == "__main__":
    try:
        main()
    except Exception:
        print(
            "=" * 88,
            file=sys.stderr,
        )
        print(
            "TRAINING FAILED",
            file=sys.stderr,
        )
        print(
            "=" * 88,
            file=sys.stderr,
        )

        if torch.cuda.is_available():
            try:
                free, total = torch.cuda.mem_get_info(
                    DEVICE
                )
                print(
                    "GPU memory at failure: "
                    f"free={free / 2**30:.2f} GiB | "
                    f"total={total / 2**30:.2f} GiB | "
                    f"allocated="
                    f"{torch.cuda.memory_allocated(DEVICE) / 2**30:.2f} GiB | "
                    f"reserved="
                    f"{torch.cuda.memory_reserved(DEVICE) / 2**30:.2f} GiB",
                    file=sys.stderr,
                )
            except Exception as memory_error:
                print(
                    "Could not collect CUDA memory summary:",
                    repr(memory_error),
                    file=sys.stderr,
                )

        print(
            "Persistent run log:",
            RUN_LOG_PATH,
            file=sys.stderr,
        )
        raise
    finally:
        if (
            "log_fp" in globals()
            and not log_fp.closed
        ):
            try:
                log_fp.flush()
            finally:
                sys.stdout = sys.__stdout__
                sys.stderr = sys.__stderr__
                log_fp.close()
                print("Log file closed.")
                print(
                    f"Log saved to: {LOG_FILE}"
                )